# BlackPearl Morphix — Stage 2, Step 3: Do knockouts change their neighbourhood?

Before training a knockout-conditioned model, check that there is **something to learn**: does the tissue around a tumour cell with knockout X look different from the tissue around a **Control** tumour cell?

**Trap avoided:** T cells and knockouts both cluster in certain places, so a knockout can look "T-cell rich" just because of *where* it sits. Every test here compares knockout cells against Control cells **from the same small region of tissue** (block-stratified permutation test), so location can't fake an effect.

**Input:** `morphix_pfish_cells.csv.gz` from Step 2 (you'll be asked to upload it if it isn't already in this session).
CPU runtime is fine; ~3–5 minutes. At the end, copy the **SUMMARY** block and paste it back.

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

import glob
cands = sorted(glob.glob('morphix_pfish_cells*.csv.gz'), key=os.path.getmtime)
if not cands:
    from google.colab import files
    print('Upload the morphix_pfish_cells file from the SUCCESSFUL Step 2 run '
          '(your browser may have named it "morphix_pfish_cells (1).csv.gz"):')
    up = files.upload()
    cands = [n for n in up if n.startswith('morphix_pfish_cells')]
FN = cands[-1]
cells = pd.read_csv(FN, index_col='cell_id')
cells['gene'] = cells['gene'].fillna('').astype(str)
print('using', FN); print(cells.cell_class.value_counts())
if not (cells.cell_class == 'tumour').any():
    raise ValueError(f'{FN} has no tumour cells: this is the file from the FAILED first Step 2 run. '
                     'Delete it (folder icon on the left, right-click > Delete) and upload the newer file '
                     '(e.g. "morphix_pfish_cells (1).csv.gz"), then run this cell again.')
S = {}

## 1. What are the 'other' cells?
'Other' mixes tumour cells with no detected guide and mouse host cells. The MERFISH probes target **human** genes, so mouse cells should have very low total counts. Check whether total counts split 'other' into two groups.

In [ ]:
tc = np.log10(cells['total_counts'] + 1)
fig, ax = plt.subplots(figsize=(8, 4))
for cls, col in [('other', 'grey'), ('tumour', 'tab:red'), ('tcell', 'tab:blue')]:
    ax.hist(tc[cells.cell_class == cls], bins=80, density=True, histtype='step', lw=2, color=col, label=cls)
ax.set_xlabel('log10(total MERFISH counts + 1)'); ax.legend(); ax.set_title('Counts per cell by class')
plt.show()
thr = np.quantile(tc[cells.cell_class == 'tumour'], 0.05)   # 95% of known tumour cells are above this
cells['nbr_class'] = cells.cell_class
cells.loc[(cells.cell_class == 'other') & (tc >= thr), 'nbr_class'] = 'other_high'   # likely unlabelled tumour
cells.loc[(cells.cell_class == 'other') & (tc < thr), 'nbr_class'] = 'other_low'     # likely mouse / host
print(f'threshold log10 counts = {thr:.2f}'); print(cells.nbr_class.value_counts())
S['other_split_threshold_log10'] = float(thr)
S['nbr_class_counts'] = cells.nbr_class.value_counts().to_dict()

## 2. Neighbourhood features around every single-knockout tumour cell
For each tumour cell with exactly one knockout (or Control), describe its 63 nearest neighbours.

In [ ]:
XY = cells[['x', 'y']].values
tree = cKDTree(XY)
d_nn, _ = tree.query(XY, k=2)
unit = float(np.median(d_nn[:, 1]))          # median cell spacing
K = 63
focal = cells[(cells.cell_class == 'tumour') & (~cells.gene.isin(['multiple', 'none', '']))].copy()
d, nb = tree.query(focal[['x', 'y']].values, k=K + 1)
d, nb = d[:, 1:] / unit, nb[:, 1:]
nc = cells.nbr_class.to_numpy(dtype=object)[nb]
focal['frac_tcell'] = (nc == 'tcell').mean(1)
focal['frac_other_low'] = (nc == 'other_low').mean(1)
focal['frac_tumour_like'] = np.isin(nc, ['tumour', 'other_high']).mean(1)
focal['patch_radius'] = d.mean(1)                              # compact vs spread neighbourhood
focal['nn_dist'] = d[:, 0]                                     # spacing to closest neighbour
d_t, _ = cKDTree(XY[cells.nbr_class.values == 'tcell']).query(focal[['x', 'y']].values, k=1)
focal['dist_to_tcell'] = d_t / unit
FEATS = ['frac_tcell', 'frac_other_low', 'frac_tumour_like', 'patch_radius', 'nn_dist', 'dist_to_tcell']
print(f'{len(focal)} focal tumour cells; cell spacing unit = {unit:.1f}')
print(focal.groupby(focal.gene == 'Control')[FEATS].mean().rename(index={True: 'Control', False: 'knockouts'}).round(3))

## 3. Each knockout vs nearby Control cells (location-controlled)
Tissue is cut into blocks of ~40 cell spacings. Within each block, knockout and Control labels are shuffled 2,000 times to see how big a difference arises by chance. Benjamini–Hochberg correction across all gene × feature tests.

In [ ]:
rng = np.random.default_rng(0)
BLOCK = 40 * unit
focal['block'] = (np.floor(focal.x / BLOCK).astype(int).astype(str) + '_' + np.floor(focal.y / BLOCK).astype(int).astype(str))
ctrl = focal[focal.gene == 'Control']
MIN_CELLS = 30
genes = [g for g, n in focal.gene.value_counts().items() if g != 'Control' and n >= MIN_CELLS]
N_PERM = 2000
rows = []
for g in genes:
    sub = focal[focal.gene.isin([g, 'Control'])]
    # keep only blocks containing both this knockout and Control cells
    ok_blocks = sub.groupby('block').gene.agg(lambda s: (s == g).any() and (s == 'Control').any())
    sub = sub[sub.block.isin(ok_blocks[ok_blocks].index)]
    is_g = (sub.gene == g).values
    if is_g.sum() < 15:
        continue
    codes = pd.factorize(sub.block.values)[0]
    base = np.argsort(codes, kind='stable')            # cells grouped by block
    X = sub[FEATS].values
    obs = X[is_g].mean(0) - X[~is_g].mean(0)
    null = np.empty((N_PERM, len(FEATS)))
    for p in range(N_PERM):
        lab = np.empty_like(is_g)
        lab[base] = is_g[np.lexsort((rng.random(len(codes)), codes))]   # shuffle labels within each block
        null[p] = X[lab].mean(0) - X[~lab].mean(0)
    pv = (1 + (np.abs(null) >= np.abs(obs)).sum(0)) / (N_PERM + 1)
    sd = X[~is_g].std(0) + 1e-9
    for f, o, pp, s_ in zip(FEATS, obs, pv, sd):
        rows.append(dict(gene=g, feature=f, n_knockout=int(is_g.sum()), n_control_same_blocks=int((~is_g).sum()),
                         diff=o, effect_size=o / s_, p=pp))
res = pd.DataFrame(rows)
# Benjamini-Hochberg
m = len(res); order = np.argsort(res.p.values)
q = np.empty(m); q[order] = np.minimum.accumulate((res.p.values[order] * m / (np.arange(m) + 1))[::-1])[::-1]
res['q'] = np.minimum(q, 1)
res = res.sort_values('p')
print(f'{len(genes)} knockouts with >= {MIN_CELLS} cells tested; {m} tests')
print(res.head(25).round(4).to_string(index=False))
res.to_csv('knockout_neighbourhood_effects.csv', index=False)

In [ ]:
sig = res[res.q < 0.1]
print(f'\nSignificant effects (q < 0.1): {len(sig)} tests across {sig.gene.nunique()} knockouts')
piv = res.pivot(index='gene', columns='feature', values='effect_size').reindex(columns=FEATS)
fig, ax = plt.subplots(figsize=(9, 0.32 * len(piv) + 1.5))
lim = np.nanmax(np.abs(piv.values))
im = ax.imshow(piv.values, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
qs = res.pivot(index='gene', columns='feature', values='q').reindex(index=piv.index, columns=FEATS)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        if qs.values[i, j] < 0.1:
            ax.text(j, i, '*', ha='center', va='center', fontsize=14)
ax.set_xticks(range(len(FEATS))); ax.set_xticklabels(FEATS, rotation=30, ha='right')
ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index)
plt.colorbar(im, label='effect size vs nearby Control (SD units)')
ax.set_title('Knockout vs Control neighbourhoods (* = q < 0.1)')
plt.tight_layout(); plt.show()

S['n_focal'] = int(len(focal)); S['cell_spacing_unit'] = unit
S['n_knockouts_tested'] = int(res.gene.nunique()); S['n_tests'] = int(m)
S['n_significant_q<0.1'] = int(len(sig)); S['significant_genes'] = sorted(sig.gene.unique().tolist())
S['top_20'] = res.head(20)[['gene', 'feature', 'n_knockout', 'diff', 'effect_size', 'p', 'q']].round(4).to_dict(orient='records')
S['p_value_histogram_counts'] = np.histogram(res.p, bins=10, range=(0, 1))[0].tolist()
try:
    from google.colab import files; files.download('knockout_neighbourhood_effects.csv')
except Exception:
    pass

## SUMMARY — copy everything printed below and paste it back

In [ ]:
print('===== SUMMARY START =====')
print(json.dumps(S, indent=1, default=str)[:15000])
print('===== SUMMARY END =====')